# ECG Model Comparison — DAT255

Deep comparison of the models trained by `training.ipynb`. This notebook
**never retrains** — it loads the saved `artifacts/` and analyses them.

**Comparisons covered:**
1. Training-curve comparison (loss, AUC, learning dynamics)
2. Standard multi-label metrics at threshold 0.5
3. Per-class metrics (precision, recall, F1) for each model
4. ROC and Precision-Recall curves
5. **Per-class threshold optimisation** — find the optimal threshold per class,
   per model (maximising F1)
6. **Soft-voting ensemble** of all three models, with threshold optimisation
7. Confusion matrices (at optimised thresholds)
8. Statistical significance (bootstrap CIs on macro-F1)
9. Grad-CAM on the best model
10. Summary table with recommendations

## 1. Load artifacts

In [ ]:
import os, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import tensorflow as tf
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, multilabel_confusion_matrix,
    accuracy_score,
)

os.environ["KERAS_BACKEND"] = "tensorflow"
ART_DIR = "artifacts"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
NUM_CLASSES = len(SUPERCLASSES)

# Load val + test sets (val is used for threshold selection, test for final eval)
data = np.load(os.path.join(ART_DIR, "eval_sets.npz"))
X_val,  y_val  = data["X_val"],  data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

# Load models
MODEL_NAMES = ["cnn", "rescnn", "lstm"]
MODEL_LABELS = {"cnn": "1D-CNN (tuned)", "rescnn": "Residual CNN", "lstm": "BiLSTM"}
models = {
    name: keras.saving.load_model(os.path.join(ART_DIR, f"ecg_{name}_final.keras"),
                                  compile=False)
    for name in MODEL_NAMES
}

# Load histories + tuner results
with open(os.path.join(ART_DIR, "histories.pkl"), "rb") as f:
    histories = pickle.load(f)

tuner_path = os.path.join(ART_DIR, "tuner_results.json")
if os.path.exists(tuner_path):
    with open(tuner_path) as f:
        tuner_results = json.load(f)
    print("Keras Tuner best hyperparameters:")
    for k, v in tuner_results.items():
        print(f"  {k:15s} = {v}")
else:
    tuner_results = None

print(f"\nTest set: {X_test.shape}   Models loaded: {list(models.keys())}")

## 2. Run inference for all models

In [ ]:
probs_val = {
    name: m.predict(X_val, batch_size=128, verbose=0)
    for name, m in models.items()
}
probs = {
    name: m.predict(X_test, batch_size=128, verbose=0)
    for name, m in models.items()
}
for name in MODEL_NAMES:
    print(f"{name:8s}  val={probs_val[name].shape}  test={probs[name].shape}")

## 3. Training curves comparison

Side-by-side training dynamics. Watch for: (a) overfitting gap between
train/val, (b) convergence speed, (c) final val-AUC.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"cnn": "#2980b9", "rescnn": "#27ae60", "lstm": "#e74c3c"}

for ax, metric in zip(axes, ["loss", "auc", "accuracy"]):
    for name, h in histories.items():
        if metric in h:
            ax.plot(h[metric], color=colors[name], linestyle="-",
                    label=f"{MODEL_LABELS[name]} train", alpha=0.8)
            ax.plot(h.get(f"val_{metric}", []), color=colors[name],
                    linestyle="--", label=f"{MODEL_LABELS[name]} val")
    ax.set_title(metric.upper()); ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig("curves_comparison.png", dpi=150); plt.show()

## 4. Standard metrics at threshold 0.5

In [ ]:
def evaluate_at_threshold(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(np.float32)
    return {
        "f1_macro":   f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro":   f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_weighted":f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc_macro":  roc_auc_score(y_true, y_prob, average="macro"),
        "auc_micro":  roc_auc_score(y_true, y_prob, average="micro"),
        "ap_macro":   average_precision_score(y_true, y_prob, average="macro"),
        "subset_acc": accuracy_score(y_true, y_pred),
    }

rows = []
for name, p in probs.items():
    r = evaluate_at_threshold(y_test, p, 0.5)
    r["model"] = MODEL_LABELS[name]
    rows.append(r)

summary_05 = pd.DataFrame(rows).set_index("model")
summary_05 = summary_05[["f1_macro","f1_micro","f1_weighted",
                         "auc_macro","auc_micro","ap_macro","subset_acc"]]
print("Metrics at threshold=0.5:")
display(summary_05.round(4))

## 5. Per-class precision / recall / F1 (threshold 0.5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, p) in zip(axes, probs.items()):
    y_pred = (p >= 0.5).astype(np.float32)
    per_class = {}
    for i, sc in enumerate(SUPERCLASSES):
        per_class[sc] = {
            "precision": f1_score(y_test[:, i], y_pred[:, i], zero_division=0,
                                  average=None).mean() if False else
                         ((y_pred[:, i] * y_test[:, i]).sum() /
                          max(y_pred[:, i].sum(), 1)),
            "recall":    (y_pred[:, i] * y_test[:, i]).sum() /
                         max(y_test[:, i].sum(), 1),
            "f1":        f1_score(y_test[:, i], y_pred[:, i], zero_division=0),
        }
    per_df = pd.DataFrame(per_class).T
    per_df.plot.bar(ax=ax, color=["#3498db", "#e67e22", "#27ae60"])
    ax.set_title(MODEL_LABELS[name]); ax.set_ylim(0, 1)
    ax.set_ylabel("Score" if ax is axes[0] else "")
    ax.grid(alpha=0.3, axis="y"); ax.legend(loc="lower right")

plt.suptitle("Per-class Precision / Recall / F1 @ threshold=0.5", y=1.02)
plt.tight_layout(); plt.savefig("per_class_metrics.png", dpi=150); plt.show()

## 6. ROC & PR curves (per class, per model)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Top row: ROC
for ax, (name, p) in zip(axes[0], probs.items()):
    for i, sc in enumerate(SUPERCLASSES):
        fpr, tpr, _ = roc_curve(y_test[:, i], p[:, i])
        ax.plot(fpr, tpr, label=f"{sc} (AUC={roc_auc_score(y_test[:, i], p[:, i]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_title(f"ROC — {MODEL_LABELS[name]}")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)

# Bottom row: PR
for ax, (name, p) in zip(axes[1], probs.items()):
    for i, sc in enumerate(SUPERCLASSES):
        prec, rec, _ = precision_recall_curve(y_test[:, i], p[:, i])
        ap = average_precision_score(y_test[:, i], p[:, i])
        ax.plot(rec, prec, label=f"{sc} (AP={ap:.3f})")
    ax.set_title(f"PR — {MODEL_LABELS[name]}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(loc="lower left", fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig("roc_pr_curves.png", dpi=150); plt.show()

## 7. Per-class threshold optimisation

The default 0.5 threshold is rarely optimal for multi-label classification
with imbalanced classes. We sweep thresholds in [0.05, 0.95] and pick the
one maximising F1 per class.

**Crucially, the threshold search is done on the validation set** (fold 9),
and the chosen thresholds are then evaluated on the held-out test set
(fold 10). This gives an unbiased estimate of the real-world lift from
threshold tuning.

In [ ]:
def find_optimal_thresholds(y_true, y_prob, grid=np.arange(0.05, 0.95, 0.01)):
    thresholds = np.zeros(y_true.shape[1])
    for i in range(y_true.shape[1]):
        f1s = [f1_score(y_true[:, i], (y_prob[:, i] >= t).astype(float),
                        zero_division=0) for t in grid]
        thresholds[i] = grid[int(np.argmax(f1s))]
    return thresholds


# Search on VAL only
optimal_thresholds = {
    name: find_optimal_thresholds(y_val, probs_val[name]) for name in MODEL_NAMES
}

print("Optimal per-class thresholds (selected on validation set):")
print(pd.DataFrame(optimal_thresholds, index=SUPERCLASSES).round(3))

In [ ]:
# Evaluate on TEST using val-selected thresholds
rows = []
for name, p in probs.items():
    y_pred_05 = (p >= 0.5).astype(np.float32)
    y_pred_opt = np.zeros_like(p)
    for i, t in enumerate(optimal_thresholds[name]):
        y_pred_opt[:, i] = (p[:, i] >= t).astype(np.float32)
    rows.append({
        "model": MODEL_LABELS[name],
        "F1 @ 0.5":        f1_score(y_test, y_pred_05, average="macro", zero_division=0),
        "F1 @ optimised":  f1_score(y_test, y_pred_opt, average="macro", zero_division=0),
        "Δ F1":            f1_score(y_test, y_pred_opt, average="macro", zero_division=0) -
                           f1_score(y_test, y_pred_05, average="macro", zero_division=0),
    })

threshold_gain = pd.DataFrame(rows).set_index("model").round(4)
print("Effect of per-class threshold optimisation on TEST macro-F1")
print("(thresholds selected on val — unbiased estimate):")
display(threshold_gain)

## 8. Soft-voting ensemble

The simplest ensemble: average the sigmoid outputs of all three models,
then apply per-class threshold optimisation. Usually gives a 1–3 F1 bump
if the models make different kinds of errors.

In [ ]:
ensemble_prob_val  = np.mean([probs_val[n] for n in MODEL_NAMES], axis=0)
ensemble_prob      = np.mean([probs[n]     for n in MODEL_NAMES], axis=0)

# Select thresholds on val
ensemble_thresholds = find_optimal_thresholds(y_val, ensemble_prob_val)

# Apply to test
y_pred_ens = np.zeros_like(ensemble_prob)
for i, t in enumerate(ensemble_thresholds):
    y_pred_ens[:, i] = (ensemble_prob[:, i] >= t).astype(np.float32)

ens_metrics = {
    "f1_macro":  f1_score(y_test, y_pred_ens, average="macro", zero_division=0),
    "f1_micro":  f1_score(y_test, y_pred_ens, average="micro", zero_division=0),
    "auc_macro": roc_auc_score(y_test, ensemble_prob, average="macro"),
    "ap_macro":  average_precision_score(y_test, ensemble_prob, average="macro"),
}
print("Soft-voting ensemble (val-selected per-class thresholds, test metrics):")
for k, v in ens_metrics.items():
    print(f"  {k:12s} = {v:.4f}")
print("Ensemble thresholds:", dict(zip(SUPERCLASSES, ensemble_thresholds.round(3))))

## 9. Confusion matrices at optimised thresholds

In [ ]:
fig, axes = plt.subplots(len(MODEL_NAMES) + 1, NUM_CLASSES,
                         figsize=(4 * NUM_CLASSES, 4 * (len(MODEL_NAMES) + 1)))

def plot_cms(y_pred, row_axes, title):
    cms = multilabel_confusion_matrix(y_test, y_pred)
    for i, (cm, sc, ax) in enumerate(zip(cms, SUPERCLASSES, row_axes)):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"], cbar=False)
        ax.set_title(f"{title} — {sc}", fontsize=10)
        if i == 0: ax.set_ylabel("True")
        ax.set_xlabel("Pred")

for row, (name, p) in zip(axes[:len(MODEL_NAMES)], probs.items()):
    y_pred = np.zeros_like(p)
    for i, t in enumerate(optimal_thresholds[name]):
        y_pred[:, i] = (p[:, i] >= t).astype(np.float32)
    plot_cms(y_pred, row, MODEL_LABELS[name])

plot_cms(y_pred_ens, axes[-1], "Ensemble")

plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

## 10. Statistical significance — bootstrap confidence intervals

Single-number metrics are misleading without error bars. We bootstrap the
test set (1000 resamples) and report 95% CIs on macro-F1 for each model.

In [ ]:
def bootstrap_f1(y_true, y_prob, thresholds, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = np.empty(n_boot)
    y_pred = np.zeros_like(y_prob)
    for i, t in enumerate(thresholds):
        y_pred[:, i] = (y_prob[:, i] >= t).astype(np.float32)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores[b] = f1_score(y_true[idx], y_pred[idx],
                             average="macro", zero_division=0)
    return scores.mean(), np.percentile(scores, 2.5), np.percentile(scores, 97.5)


rows = []
for name, p in probs.items():
    mean, lo, hi = bootstrap_f1(y_test, p, optimal_thresholds[name])
    rows.append({"model": MODEL_LABELS[name], "F1 mean": mean,
                 "95% CI low": lo, "95% CI high": hi})
mean, lo, hi = bootstrap_f1(y_test, ensemble_prob, ensemble_thresholds)
rows.append({"model": "Ensemble", "F1 mean": mean,
             "95% CI low": lo, "95% CI high": hi})

boot_df = pd.DataFrame(rows).set_index("model").round(4)
print("Bootstrap 95% CIs (1000 resamples):")
display(boot_df)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(boot_df.index, boot_df["F1 mean"],
            yerr=[boot_df["F1 mean"] - boot_df["95% CI low"],
                  boot_df["95% CI high"] - boot_df["F1 mean"]],
            fmt="o", capsize=5, markersize=8)
ax.set_ylabel("Macro F1"); ax.set_title("Bootstrap 95% CI on Macro F1")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("bootstrap_ci.png", dpi=150); plt.show()

## 11. Grad-CAM on the best single model

Pick the model with the highest bootstrap mean F1 (excluding ensemble)
and run Grad-CAM on a few test samples.

In [ ]:
# Pick best single model
single_scores = boot_df.drop("Ensemble")
best_label = single_scores["F1 mean"].idxmax()
best_name  = {v: k for k, v in MODEL_LABELS.items()}[best_label]
best_model = models[best_name]
print(f"Best single model: {best_label}")

# Identify last conv layer name
last_conv_name = None
for layer in reversed(best_model.layers):
    if isinstance(layer, keras.layers.Conv1D):
        last_conv_name = layer.name
        break
print(f"Last conv layer for Grad-CAM: {last_conv_name}")

In [ ]:
def compute_gradcam_1d(model, input_sample, class_idx, last_conv_name):
    grad_model = keras.Model(inputs=model.input,
                             outputs=[model.get_layer(last_conv_name).output,
                                      model.output])
    x = tf.cast(input_sample, tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x)
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=1)
    cam = tf.reduce_sum(conv_out * weights[:, tf.newaxis, :], axis=-1)
    cam = tf.nn.relu(cam).numpy().squeeze()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    # Upsample to 1000 samples
    cam = np.interp(np.linspace(0, len(cam) - 1, 1000),
                    np.arange(len(cam)), cam)
    return cam


# Pick samples with clear positive labels for each non-NORM class
LEAD_II = 1
for cls in [1, 2, 3, 4]:  # MI, STTC, CD, HYP
    candidates = np.where(y_test[:, cls] == 1)[0]
    if len(candidates) == 0: continue
    idx = candidates[0]
    sample = X_test[idx:idx+1]
    heatmap = compute_gradcam_1d(best_model, sample, cls, last_conv_name)

    fig, ax = plt.subplots(figsize=(12, 3))
    t = np.arange(1000) / 100
    ax.plot(t, sample[0, :, LEAD_II], color="#2c3e50", linewidth=0.8)
    ax.imshow(heatmap[np.newaxis, :], cmap="Reds", alpha=0.4,
              aspect="auto", extent=[t[0], t[-1], *ax.get_ylim()], origin="lower")
    ax.set_title(f"Grad-CAM for {SUPERCLASSES[cls]} — {best_label} (sample {idx}, Lead II)")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude (normalised)")
    plt.tight_layout(); plt.show()

## 12. Final summary

In [ ]:
print("=" * 70)
print("  FINAL COMPARISON SUMMARY")
print("=" * 70)

if tuner_results:
    print("\n[Keras Tuner winning config for CNN]")
    for k, v in tuner_results.items():
        print(f"  {k:15s} = {v}")

print("\n[Bootstrap macro-F1 ranking]")
ranking = boot_df.sort_values("F1 mean", ascending=False)
for i, (name, row) in enumerate(ranking.iterrows(), 1):
    print(f"  {i}. {name:20s}  F1 = {row['F1 mean']:.4f}  "
          f"[{row['95% CI low']:.4f}, {row['95% CI high']:.4f}]")

print("\n[Threshold optimisation gain]")
display(threshold_gain)

print("\n[Recommendation]")
winner = ranking.index[0]
print(f"  Winning model: {winner}")
print(f"  Deploy this to HF Space; update app.py to load the corresponding .keras file.")
print(f"  Use per-class thresholds rather than 0.5 for classification decisions.")
print("=" * 70)